# ABINet (Booster) - OCR Model Testing

**Model**: TrOCR Fine-tuned for Handwritten Text Recognition
**Dataset**: Custom validation set with 41,370 samples
**Metrics**: Accuracy (case-insensitive), CER, WER, Similarity

In [ ]:
# Setup: Load model, processor, dataset, and metric functions

import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from transformers import VisionEncoderDecoderModel, TrOCRProcessor
from Levenshtein import distance as edit_distance
import difflib
import time

print("✅ Imports completed")

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Define metrics (case-insensitive)
def accuracy_score(gt, pred):
    """Exact match accuracy (case-insensitive)"""
    return 1.0 if gt.upper() == pred.upper() else 0.0

def cer_score(gt, pred):
    """Character Error Rate (case-insensitive)"""
    gt = gt.upper()
    pred = pred.upper()
    if len(gt) == 0:
        return 0 if len(pred) == 0 else 1
    return edit_distance(gt, pred) / len(gt)

def wer_score(gt, pred):
    """Word Error Rate (case-insensitive)"""
    gt_words = gt.upper().split()
    pred_words = pred.upper().split()
    if len(gt_words) == 0:
        return 0 if len(pred_words) == 0 else 1
    return edit_distance(gt_words, pred_words) / len(gt_words)

def similarity_score(gt, pred):
    """Sequence Similarity (case-insensitive)"""
    gt = gt.upper()
    pred = pred.upper()
    return difflib.SequenceMatcher(None, gt, pred).ratio()

print("✅ Metrics defined")

# Load model and processor
model_path = "best_ocr_model"
try:
    model = VisionEncoderDecoderModel.from_pretrained(model_path)
    processor = TrOCRProcessor.from_pretrained(model_path)
    model.to(device)
    model.eval()
    print(f"✅ Model loaded: {model_path}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    model = None
    processor = None

# Load validation dataset
dataset_path = Path("ocr_dataset")
validation_csv = dataset_path / "written_name_validation_v2.csv"
validation_dir = dataset_path / "validation_v2" / "validation"

try:
    val_df = pd.read_csv(validation_csv)
    print(f"✅ Dataset loaded: {len(val_df)} samples")
    print(f"   CSV: {validation_csv}")
    print(f"   Images: {validation_dir}")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    val_df = None

In [ ]:
# Test on a single random image

def test_single_image(image_idx=None):
    """Test model on a single random or specified image"""
    if image_idx is None:
        image_idx = np.random.randint(0, len(val_df))
    
    row = val_df.iloc[image_idx]
    image_path = validation_dir / row['FILENAME']
    ground_truth = row['IDENTITY']
    
    if not image_path.exists():
        print(f"❌ Image not found: {image_path}")
        return None
    
    # Load and predict
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_length=128)
    
    prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # Calculate metrics
    acc = accuracy_score(ground_truth, prediction)
    cer = cer_score(ground_truth, prediction)
    
    # Display
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.imshow(image)
    ax.axis('off')
    status = "✅" if acc == 1.0 else "❌"
    title = f"{status} Pred: {prediction} | GT: {ground_truth} | Acc: {acc:.0%}"
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.show()
    
    return {"pred": prediction, "truth": ground_truth, "acc": acc, "cer": cer}

# Run test
result = test_single_image()
if result:
    print(f"Prediction: {result['pred']}")
    print(f"Ground Truth: {result['truth']}")
    print(f"Accuracy: {result['acc']:.0%} | CER: {result['cer']:.4f}")

In [ ]:

import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from transformers import VisionEncoderDecoderModel, TrOCRProcessor
from Levenshtein import distance as edit_distance
import difflib
import time
# Batch test on 10 random images

def test_batch(num_samples=10):
    """Test on multiple random images"""
    results = []
    random_indices = np.random.choice(len(val_df), min(num_samples, len(val_df)), replace=False)
    
    print(f"Testing {len(random_indices)} random images...\n")
    
    for i, idx in enumerate(random_indices):
        row = val_df.iloc[idx]
        image_path = validation_dir / row['FILENAME']
        ground_truth = row['IDENTITY']
        
        if not image_path.exists():
            continue
        
        try:
            image = Image.open(image_path).convert("RGB")
            pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
            
            with torch.no_grad():
                generated_ids = model.generate(pixel_values, max_length=128)
            
            prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            acc = accuracy_score(ground_truth, prediction)
            cer = cer_score(ground_truth, prediction)
            
            results.append({"pred": prediction, "truth": ground_truth, "acc": acc, "cer": cer})
            
            status = "✅" if acc == 1.0 else "❌"
            print(f"{status} [{i+1}/{len(random_indices)}] {prediction:20s} | {ground_truth:20s} | {acc:.0%}")
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # Summary
    if results:
        accs = [r["acc"] for r in results]
        cers = [r["cer"] for r in results]
        print(f"\n{'='*60}")
        print(f"Accuracy: {np.mean(accs):.2%} | CER: {np.mean(cers):.4f}")
        print(f"Correct: {sum(1 for a in accs if a==1.0)}/{len(accs)}")
        print(f"{'='*60}")
    
    return results

# Run batch test
batch_results = test_batch(num_samples=10)

# **Image**

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
model = AutoModelForImageTextToText.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

c:\Users\trilokesh.sarkar\Desktop\Oce_Custom_Model\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\trilokesh.sarkar\.cache\huggingface\hub\models--Qwen--Qwen2.5-VL-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
